In [ ]:
# yolox-ppe-cpu: build the Rust detector and serve it behind an
# OpenAI-compatible /v1/chat/completions endpoint (single Colab cell).
# Upload your OpenVINO IR (yolox_ppe.xml + .bin) or ONNX into /content first.
import base64, glob, io, json, os, subprocess, sys, tempfile, threading, time, urllib.request, uuid

REPO = "https://github.com/dhaneshkk/cpuinfer.git"
BRANCH = "claude/yolox-ppe-rust-xhiwfu"
WORK = "/content/cpuinfer"
MODEL_XML = "/content/ir_model/yolox_ppe.xml"
ONNX = "/content/yolox_ppe.onnx"          # converted to IR if the .xml is missing
NUM_CLASSES, INPUT_SIZE, PORT = 3, 416, 8000
CLASS_NAMES = ["helmet", "vest", "person"]  # edit to match your training labels

def sh(cmd, **kw):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True, **kw)

# 1. Python deps: OpenVINO wheel (also provides libopenvino_c.so for Rust), server.
sh(f"{sys.executable} -m pip -q install openvino fastapi uvicorn pillow requests")
import openvino as ov

# 2. Rust toolchain + build.
cargo = os.path.expanduser("~/.cargo/bin/cargo")
if not os.path.exists(cargo):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
if not os.path.isdir(WORK):
    sh(f"git clone --depth 1 -b {BRANCH} {REPO} {WORK}")
sh(f"{cargo} build --release", cwd=WORK)
BIN = f"{WORK}/target/release/yolox-ppe-cpu"

# 3. Point openvino-rs (runtime linking) at the wheel's C library.
libdir = os.path.join(os.path.dirname(ov.__file__), "libs")
c_lib = sorted(glob.glob(f"{libdir}/libopenvino_c.so*"))[0]
link_dir = "/content/ovlib"; os.makedirs(link_dir, exist_ok=True)
link = f"{link_dir}/libopenvino_c.so"
if not os.path.exists(link): os.symlink(c_lib, link)
ENV = dict(os.environ, LD_LIBRARY_PATH=f"{link_dir}:{libdir}:" + os.environ.get("LD_LIBRARY_PATH", ""),
           OPENVINO_LIB_PATH=link_dir)

# 4. Model: convert ONNX -> IR with a fixed 1x3xSxS input (no dynamic axes).
if not os.path.exists(MODEL_XML):
    assert os.path.exists(ONNX), f"Upload {MODEL_XML} (+.bin) or {ONNX}"
    os.makedirs(os.path.dirname(MODEL_XML), exist_ok=True)
    ov.save_model(ov.convert_model(ONNX, input=[1, 3, INPUT_SIZE, INPUT_SIZE]), MODEL_XML)

def detect(image_bytes, score=0.3, iou=0.45):
    with tempfile.NamedTemporaryFile(suffix=".png") as f:
        from PIL import Image
        Image.open(io.BytesIO(image_bytes)).convert("L").save(f.name)
        out = subprocess.run([BIN, "--model", MODEL_XML, "--image", f.name, "--size", str(INPUT_SIZE),
                              "--classes", str(NUM_CLASSES), "--score", str(score), "--iou", str(iou)],
                             capture_output=True, text=True, env=ENV)
    if out.returncode != 0:
        raise RuntimeError(out.stderr.strip())
    if "[WARN]" in out.stderr: print(out.stderr.strip())
    return json.loads(out.stdout)["detections"]

# 5. OpenAI-compatible server. The image goes in a user message as an
#    `image_url` content part (data: URL or http(s) URL); the reply's content
#    is the detections as JSON.
from fastapi import FastAPI, HTTPException
import uvicorn

app = FastAPI(title="yolox-ppe-cpu")
MODEL_ID = "yolox-ppe-cpu"

def image_from_messages(messages):
    for m in reversed(messages):
        parts = m.get("content")
        for p in parts if isinstance(parts, list) else []:
            if p.get("type") == "image_url":
                url = p["image_url"]["url"] if isinstance(p["image_url"], dict) else p["image_url"]
                if url.startswith("data:"):
                    return base64.b64decode(url.split(",", 1)[1])
                with urllib.request.urlopen(url, timeout=20) as r:
                    return r.read()
    raise HTTPException(400, "No image_url content part in messages")

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "local"}]}

@app.post("/v1/chat/completions")
def chat(req: dict):
    try:
        dets = detect(image_from_messages(req.get("messages", [])),
                      score=float(req.get("score_threshold", 0.3)), iou=float(req.get("iou_threshold", 0.45)))
    except RuntimeError as e:
        raise HTTPException(500, str(e))
    for d in dets:
        d["label"] = CLASS_NAMES[d["class_id"]] if d["class_id"] < len(CLASS_NAMES) else str(d["class_id"])
    content = json.dumps({"detections": dets})
    return {"id": f"chatcmpl-{uuid.uuid4().hex}", "object": "chat.completion", "created": int(time.time()),
            "model": req.get("model", MODEL_ID),
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant", "content": content}}],
            "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}}

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started: time.sleep(0.2)

# 6. Smoke test with a synthetic grey frame through the OpenAI client format.
import requests
from PIL import Image
buf = io.BytesIO(); Image.new("L", (1280, 720), 128).save(buf, "PNG")
payload = {"model": MODEL_ID, "messages": [{"role": "user", "content": [
    {"type": "text", "text": "Detect PPE"},
    {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}}]}]}
r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", json=payload, timeout=120)
print(r.status_code, r.json()["choices"][0]["message"]["content"] if r.ok else r.text)
print(f"Serving OpenAI-compatible API at http://127.0.0.1:{PORT}/v1 (use base_url with the openai client)")
